# Leak-Free Target Encoding

This experiment adds target-derived statistical features without
allowing validation targets to influence their own features.

Target encoding will be generated out-of-fold for training rows.

Validation encodings are calculated using training-fold targets only.

No test or validation target information is used.

In [7]:
import pandas as pd
import numpy as np

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.metrics import roc_auc_score

In [8]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

y = train["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

train.shape, test.shape

((668665, 15), (286571, 14))

In [2]:
import pandas as pd
import numpy as np

from lightgbm import LGBMClassifier, early_stopping, log_evaluation

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.metrics import roc_auc_score

In [10]:
def add_features(df):
    df = df.copy()

    # Numerical interactions
    df["Income_per_Car"] = (
        df["Annual_Income_USD"]
        / df["Number_of_Cars_Owned"].clip(lower=1)
    )

    df["Total_Charging_Stations"] = (
        df["Charging_Stations_Near_Home"]
        + df["Charging_Stations_Near_Work"]
    )

    df["Charging_Station_Difference"] = (
        df["Charging_Stations_Near_Home"]
        - df["Charging_Stations_Near_Work"]
    )

    df["Income_x_Concern"] = (
        df["Annual_Income_USD"]
        * df["Environmental_Concern_Level"]
    )

    df["Commute_x_Concern"] = (
        df["Daily_Commute_km"]
        * df["Environmental_Concern_Level"]
    )

    # Categorical interactions
    df["Subsidy_x_Concern"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Environmental_Concern_Level"].astype(str)
    )

    df["Subsidy_x_Anxiety"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    df["Concern_x_Anxiety"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    income_band = pd.cut(
        df["Annual_Income_USD"],
        bins=[
            -np.inf,
            60000,
            80000,
            100000,
            120000,
            np.inf
        ],
        labels=[
            "Income_1",
            "Income_2",
            "Income_3",
            "Income_4",
            "Income_5"
        ]
    ).astype(str)

    df["Income_x_Subsidy"] = (
        income_band
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    df["HomeCharge_x_Subsidy"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    commute_band = pd.cut(
        df["Daily_Commute_km"],
        bins=[
            -np.inf,
            20,
            40,
            60,
            np.inf
        ],
        labels=[
            "Short",
            "Medium",
            "Long",
            "VeryLong"
        ]
    ).astype(str)

    df["HomeCharge_x_Commute"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + commute_band
    )

    return df

In [12]:
train_fe = add_features(train)
test_fe = add_features(test)

train_fe.shape, test_fe.shape

((668665, 26), (286571, 25))

In [14]:
def make_te_keys(df):
    keys = pd.DataFrame(index=df.index)

    keys["Subsidy"] = (
        df["Subsidy_Available"].astype(str)
    )

    keys["Concern"] = (
        df["Environmental_Concern_Level"].astype(str)
    )

    keys["Anxiety"] = (
        df["Range_Anxiety_Level"].astype(str)
    )

    keys["HomeCharge"] = (
        df["Home_Charging_Possible"].astype(str)
    )

    keys["City"] = (
        df["City_Type"].astype(str)
    )

    keys["CarType"] = (
        df["Current_Car_Type"].astype(str)
    )

    keys["Subsidy_Concern"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Environmental_Concern_Level"].astype(str)
    )

    keys["Subsidy_Anxiety"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    keys["Concern_Anxiety"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    income_band = pd.cut(
        df["Annual_Income_USD"],
        bins=[
            -np.inf,
            60000,
            80000,
            100000,
            120000,
            np.inf
        ],
        labels=[
            "Income_1",
            "Income_2",
            "Income_3",
            "Income_4",
            "Income_5"
        ]
    ).astype(str)

    keys["IncomeBand_Subsidy"] = (
        income_band
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    keys["IncomeBand_Concern"] = (
        income_band
        + "_"
        + df["Environmental_Concern_Level"].astype(str)
    )

    keys["HomeCharge_Subsidy"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    return keys

In [15]:
te_keys = make_te_keys(train)

te_keys.head()

,Subsidy,Concern,Anxiety,HomeCharge,City,CarType,Subsidy_Concern,Subsidy_Anxiety,Concern_Anxiety,IncomeBand_Subsidy,IncomeBand_Concern,HomeCharge_Subsidy
0,No,1.0,Low,Yes,Suburban,Sedan,No_1.0,No_Low,1.0_Low,Income_3_No,Income_3_1.0,Yes_No
1,No,4.0,Low,Yes,Rural,SUV,No_4.0,No_Low,4.0_Low,Income_1_No,Income_1_4.0,Yes_No
2,Yes,5.0,Low,No,Urban,Sedan,Yes_5.0,Yes_Low,5.0_Low,Income_3_Yes,Income_3_5.0,No_Yes
3,No,3.0,Low,Yes,Suburban,Hatchback,No_3.0,No_Low,3.0_Low,Income_2_No,Income_2_3.0,Yes_No
4,No,3.0,Low,Yes,Suburban,Hatchback,No_3.0,No_Low,3.0_Low,Income_1_No,Income_1_3.0,Yes_No


In [16]:
def build_smoothed_mapping(
    key,
    target,
    smoothing=50
):
    temp = pd.DataFrame({
        "key": key.astype(str).to_numpy(),
        "target": target.to_numpy()
    })

    global_mean = target.mean()

    stats = (
        temp
        .groupby("key")["target"]
        .agg(["sum", "count"])
    )

    stats["encoding"] = (
        stats["sum"]
        + smoothing * global_mean
    ) / (
        stats["count"]
        + smoothing
    )

    return (
        stats["encoding"],
        global_mean
    )

In [17]:
def create_oof_target_encodings(
    train_df,
    valid_df,
    y_train,
    smoothing=50,
    n_splits=5,
    random_state=42
):
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    train_keys = make_te_keys(train_df)
    valid_keys = make_te_keys(valid_df)

    train_te = pd.DataFrame(
        index=train_df.index
    )

    valid_te = pd.DataFrame(
        index=valid_df.index
    )

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for column in train_keys.columns:

        oof_values = np.zeros(
            len(train_df)
        )

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            train_df,
            y_train
        ):

            mapping, global_mean = build_smoothed_mapping(
                train_keys[column].iloc[inner_train_idx],
                y_train.iloc[inner_train_idx],
                smoothing=smoothing
            )

            encoded = (
                train_keys[column]
                .iloc[inner_valid_idx]
                .astype(str)
                .map(mapping)
                .fillna(global_mean)
            )

            oof_values[inner_valid_idx] = encoded

        train_te[
            f"TE_{column}"
        ] = oof_values

        # For the true validation set, mapping is learned
        # from the entire outer training set only.
        full_mapping, full_global_mean = build_smoothed_mapping(
            train_keys[column],
            y_train,
            smoothing=smoothing
        )

        valid_te[
            f"TE_{column}"
        ] = (
            valid_keys[column]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

    return train_te, valid_te

In [18]:
train_idx, valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [19]:
outer_train_raw = train.iloc[train_idx].copy()
outer_valid_raw = train.iloc[valid_idx].copy()

y_outer_train = y.iloc[train_idx].copy()
y_outer_valid = y.iloc[valid_idx].copy()

In [20]:
train_te, valid_te = create_oof_target_encodings(
    outer_train_raw,
    outer_valid_raw,
    y_outer_train,
    smoothing=50,
    n_splits=5,
    random_state=42
)

In [21]:
train_te.shape, valid_te.shape

((534932, 12), (133733, 12))

In [22]:
X_base = train_fe.drop(
    columns=["id", "Will_Buy_EV"]
)

X_train_te = (
    X_base
    .iloc[train_idx]
    .reset_index(drop=True)
)

X_valid_te = (
    X_base
    .iloc[valid_idx]
    .reset_index(drop=True)
)

In [23]:
X_train_te = pd.concat(
    [
        X_train_te,
        train_te
    ],
    axis=1
)

X_valid_te = pd.concat(
    [
        X_valid_te,
        valid_te
    ],
    axis=1
)

In [24]:
X_train_te.shape, X_valid_te.shape

((534932, 36), (133733, 36))

In [25]:
categorical_cols_te = X_train_te.select_dtypes(
    include=["object", "category"]
).columns.tolist()

/tmp/ipykernel_12911/3331692004.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_te = X_train_te.select_dtypes(


In [26]:
for col in categorical_cols_te:
    X_train_te[col] = X_train_te[col].astype("category")
    X_valid_te[col] = X_valid_te[col].astype("category")

In [27]:
lgb_te_model = LGBMClassifier(
    objective="binary",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [28]:
lgb_te_model.fit(
    X_train_te,
    y_outer_train.reset_index(drop=True),

    eval_X=X_valid_te,
    eval_y=y_outer_valid.reset_index(drop=True),

    eval_metric="auc",

    callbacks=[
        early_stopping(
            150,
            first_metric_only=True
        ),
        log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.940895	valid_0's binary_logloss: 0.228206
[400]	valid_0's auc: 0.941512	valid_0's binary_logloss: 0.227035
[600]	valid_0's auc: 0.941703	valid_0's binary_logloss: 0.226694
[800]	valid_0's auc: 0.941741	valid_0's binary_logloss: 0.226613
[1000]	valid_0's auc: 0.941768	valid_0's binary_logloss: 0.226552
Early stopping, best iteration is:
[998]	valid_0's auc: 0.941771	valid_0's binary_logloss: 0.226548
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [29]:
te_valid_pred = lgb_te_model.predict_proba(
    X_valid_te
)[:, 1]

te_auc = roc_auc_score(
    y_outer_valid,
    te_valid_pred
)

te_auc

0.9417705187404212

In [30]:
print(f"Raw LightGBM:          {0.941669:.6f}")
print(f"Feature Engineering:   {0.941785:.6f}")
print(f"FE + Target Encoding:  {te_auc:.6f}")

print()
print(
    f"Gain vs raw: "
    f"{te_auc - 0.941669:+.6f}"
)

print(
    f"Gain vs FE:  "
    f"{te_auc - 0.941785:+.6f}"
)

print()
print(
    f"Best iteration: "
    f"{lgb_te_model.best_iteration_}"
)

Raw LightGBM:          0.941669
Feature Engineering:   0.941785
FE + Target Encoding:  0.941771

Gain vs raw: +0.000102
Gain vs FE:  -0.000014

Best iteration: 998
